In [ ]:
from IPython.display import HTML, display
display(HTML('''
<div style="text-align:right;padding:8px 0;">
<button id="toggle-code-btn" style="
  padding:8px 18px;font-size:14px;cursor:pointer;
  background:#4a90d9;color:white;border:none;border-radius:5px;
  box-shadow:0 2px 4px rgba(0,0,0,0.2);
" onclick="
  var cells = document.querySelectorAll('.jp-Cell-inputWrapper');
  var btn = document.getElementById('toggle-code-btn');
  var show = btn.dataset.show !== 'true';
  btn.dataset.show = show;
  btn.textContent = show ? '\u9690\u85cf\u4ee3\u7801 \ud83d\udd12' : '\u663e\u793a\u4ee3\u7801 \ud83d\udd13';
  cells.forEach(function(c){ c.style.display = show ? '' : 'none'; });
">\u663e\u793a\u4ee3\u7801 \ud83d\udd13</button>
</div>
'''))


# 03 - 矢量图形：用数学画画

**Cambridge International AS & A Level Computer Science (9618) - Section 1.2**

---

> "位图是用像素点'描述'图像，矢量图是用数学方程'定义'图像。一个是照片，一个是蓝图。"

本节将深入理解：
- 矢量图形的工作原理
- 绘图列表 (Drawing List) 是什么
- 为什么矢量图放大不失真
- 位图 vs 矢量图的详细对比
- 什么场景该用哪种

In [ ]:
%matplotlib inline

# === 中文字体配置 (Chinese Font Setup) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

# 方法1: 全局设置 WenQuanYi Zen Hei 字体
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 方法2: 强制重建字体缓存（首次运行可能需要）
fm._load_fontmanager(try_read_cache=False)

# 验证
test_font = fm.findfont('WenQuanYi Zen Hei')
if 'WenQuanYi Zen Hei' in test_font or 'wqy' in test_font.lower():
    print('✅ 中文字体 WenQuanYi Zen Hei 已加载')
else:
    print(f'⚠️ WenQuanYi Zen Hei 未找到，使用: {test_font}')
    # Fallback: 直接注册字体文件
    font_path = '/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc'
    if __import__('os').path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei'] + plt.rcParams['font.sans-serif']
        print('✅ 已手动加载 WenQuanYi Zen Hei 字体文件')

import sys
try:
    import matplotlib.patches as patches
    from matplotlib.colors import ListedColormap
    import numpy as np
    from IPython.display import display, HTML, Image
    print('Libraries loaded!')
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'matplotlib', 'numpy', '-q'])
    import matplotlib.patches as patches
    from matplotlib.colors import ListedColormap
    import numpy as np
    from IPython.display import display, HTML, Image
    print('Installed & loaded!')
plt.rcParams.update({'font.size': 12, 'figure.figsize': (10,6),
    'font.sans-serif': ['WenQuanYi Zen Hei','Noto Sans CJK SC','DejaVu Sans'], 'axes.unicode_minus': False})


---
## 1. 两种截然不同的思路

### 位图的思路：逐像素描述
"在第 (0,0) 位置放一个红色点，(0,1) 放一个红色点，(0,2) 放一个蓝色点..."

就像用马赛克瓷砖拼图——每一块砖的颜色都要单独记录。

### 矢量图的思路：用数学描述
"画一个圆：圆心在 (5,5)，半径 3，边框红色 2px，填充蓝色。"

就像给画家写了一份指令——告诉他画什么形状、什么颜色，画家自己去执行。

**关键区别：**
- 位图存储的是 **"结果"**（每个点的颜色）
- 矢量图存储的是 **"指令"**（怎么画出来的）

In [ ]:
# 对比演示
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: Bitmap approach
bitmap = np.zeros((10, 10, 3))
# Draw a rough circle with pixels
for i in range(10):
    for j in range(10):
        if (i-4.5)**2 + (j-4.5)**2 < 12:
            bitmap[i, j] = [0.2, 0.4, 0.9]  # Blue fill
        if abs((i-4.5)**2 + (j-4.5)**2 - 12) < 4:
            bitmap[i, j] = [0.9, 0.1, 0.1]  # Red border

ax1.imshow(bitmap, interpolation='nearest')
ax1.set_title('Bitmap Approach\nStores colour of EVERY pixel', fontsize=13, fontweight='bold')
ax1.set_xticks(np.arange(-0.5, 10, 1), minor=True)
ax1.set_yticks(np.arange(-0.5, 10, 1), minor=True)
ax1.grid(which='minor', color='white', linewidth=1)
ax1.set_xticks([]); ax1.set_yticks([])

# Data representation
data_txt = "Pixel data (100 values):\n"
data_txt += "(0,0)=[0,0,0] (0,1)=[0,0,0]\n"
data_txt += "(0,2)=[0,0,0] (0,3)=[230,25,25]\n"
data_txt += "(0,4)=[230,25,25] ... (99 more)\n"
data_txt += "\nStorage: 100 x 3 = 300 bytes"
ax1.text(5, 11.5, data_txt, ha='center', fontsize=9, fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow'))

# Right: Vector approach
ax2.set_xlim(0, 10); ax2.set_ylim(0, 10); ax2.set_aspect('equal')
circle = patches.Circle((5, 5), 3.5, facecolor='#3366EE', edgecolor='red', linewidth=3)
ax2.add_patch(circle)
ax2.set_title('Vector Approach\nStores mathematical INSTRUCTIONS', fontsize=13, fontweight='bold')
ax2.set_facecolor('#f0f0f0')
ax2.set_xticks([]); ax2.set_yticks([])

vec_txt = "Drawing List:\n"
vec_txt += "  Circle(\n"
vec_txt += "    centre=(5,5),\n"
vec_txt += "    radius=3.5,\n"
vec_txt += "    border=red 3px,\n"
vec_txt += "    fill=#3366EE\n"
vec_txt += "  )\n"
vec_txt += "\nStorage: ~50 bytes"
ax2.text(5, -2.5, vec_txt, ha='center', fontsize=9, fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightcyan'))

plt.tight_layout()
plt.show()

print('The bitmap needs 300 bytes for a 10x10 rough circle.')
print('The vector needs ~50 bytes for a PERFECT circle at ANY size!')
print('This is why vector files are usually MUCH smaller for geometric shapes.')

---
## 2. 绘图列表 (Drawing List)

每个矢量图形文件包含一个 **绘图列表**，列出所有要画的对象及其 **属性 (Attributes)**。

### 常见属性

| 属性 | 说明 | 例子 |
|:---|:---|:---|
| **位置** | 对象的 x, y 坐标 | centre=(100, 200) |
| **尺寸** | 宽、高、半径等 | radius=50, width=200 |
| **线条颜色** | 边框/轮廓的颜色 | stroke=#FF0000 |
| **线条粗细** | 边框的像素宽度 | stroke-width=2 |
| **线条样式** | 实线、虚线等 | stroke-dasharray=5,3 |
| **填充颜色** | 内部的颜色 | fill=#00FF00 |
| **旋转角度** | 旋转多少度 | rotate=45 |
| **层级顺序** | 谁在前谁在后 | z-index=1 |

### SVG 格式示例

实际的矢量图形格式（如 SVG）长这样：

```xml
<svg width="200" height="200">
  <circle cx="100" cy="100" r="50"
          fill="blue" stroke="red" stroke-width="3"/>
  <rect x="10" y="10" width="80" height="40"
        fill="green" stroke="black" stroke-width="1"/>
</svg>
```

本质上就是一个 **文本文件**！这也是为什么 SVG 文件通常很小。

In [ ]:
# 构建一个矢量图形，展示 Drawing List
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

# Left: The vector image
ax1.set_xlim(0, 200); ax1.set_ylim(0, 200); ax1.set_aspect('equal')
ax1.set_facecolor('#f5f5f5')

# Drawing list items
shapes = [
    ('Rectangle', patches.Rectangle((20, 120), 160, 60, facecolor='#4CAF50', edgecolor='#2E7D32', linewidth=2)),
    ('Circle', patches.Circle((100, 70), 40, facecolor='#2196F3', edgecolor='#0D47A1', linewidth=2)),
    ('Ellipse', patches.Ellipse((100, 150), 80, 30, facecolor='#FF9800', edgecolor='#E65100', linewidth=2)),
    ('Triangle', plt.Polygon([(60,20),(140,20),(100,60)], facecolor='#F44336', edgecolor='#B71C1C', linewidth=2)),
]

for name, shape in shapes:
    ax1.add_patch(shape)

ax1.set_title('Vector Image', fontsize=14, fontweight='bold')
ax1.set_xlabel('Each shape is an object with attributes', fontsize=10)

# Right: The Drawing List
drawing_list = "DRAWING LIST:\n---------------------------------\n1. Rectangle\n   position: (20, 120)\n   width: 160, height: 60\n   fill: #4CAF50 (green)\n   border: #2E7D32, 2px\n\n2. Circle\n   centre: (100, 70)\n   radius: 40\n   fill: #2196F3 (blue)\n   border: #0D47A1, 2px\n\n3. Ellipse\n   centre: (100, 150)\n   rx: 40, ry: 15\n   fill: #FF9800 (orange)\n   border: #E65100, 2px\n\n4. Triangle\n   points: (60,20)(140,20)(100,60)\n\n   fill: #F44336 (red)\n   border: #B71C1C, 2px\n---------------------------------\nTotal: 4 objects, ~200 bytes\n(Bitmap equivalent: ~120,000 bytes!)"

ax2.text(0.05, 0.5, drawing_list, transform=ax2.transAxes, fontsize=10,
         fontfamily='monospace', verticalalignment='center',
         bbox=dict(boxstyle='round,pad=0.8', facecolor='lightyellow', edgecolor='orange', linewidth=2))
ax2.axis('off')
ax2.set_title('Drawing List (what the file stores)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 3. 为什么矢量图放大不失真？

这是矢量图最大的优势，原理其实很简单：

### 位图放大时：
每个像素变成更大的方块 -> 你能看到一个个方块 -> **像素化**

### 矢量图放大时：
数学指令不变（"画一个半径 50 的圆"）-> 软件用更多像素重新渲染 -> **永远清晰**

**类比：**
- 位图就像一张纸上的印刷图——放大镜下能看到墨点
- 矢量图就像一个数学公式——不管算到多少位小数，结果都是精确的

In [ ]:
# 缩放对比演示
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Top: Vector (always sharp)
for ax, zoom in zip(axes[0], [1, 2, 4, 8]):
    ax.set_xlim(-1/zoom, 1/zoom); ax.set_ylim(-1/zoom, 1/zoom)
    ax.add_patch(plt.Circle((0, 0), 0.5, fill=False, color='blue', linewidth=2))
    ax.add_patch(plt.Circle((0, 0), 0.3, facecolor='lightblue', edgecolor='none'))
    ax.set_aspect('equal')
    ax.set_title(f'Vector {zoom}x', fontsize=12, fontweight='bold', color='blue')
    ax.set_xticks([]); ax.set_yticks([])

# Bottom: Bitmap (gets pixelated)
R = 16
y, x = np.mgrid[-1:1:complex(R), -1:1:complex(R)]
bmp = np.zeros((R, R, 3))
mask_fill = x**2 + y**2 < 0.3**2
mask_border = (x**2 + y**2 < 0.5**2) & ~(x**2 + y**2 < 0.4**2)
bmp[mask_fill] = [0.7, 0.85, 1.0]
bmp[mask_border] = [0, 0, 1]

specs = [(0,R,0,R), (R//4,3*R//4,R//4,3*R//4), (3*R//8,5*R//8,3*R//8,5*R//8), (7*R//16,9*R//16,7*R//16,9*R//16)]
for ax, (r1,r2,c1,c2), zoom in zip(axes[1], specs, [1,2,4,8]):
    crop = bmp[r1:r2, c1:c2]
    ax.imshow(crop, interpolation='nearest')
    ax.set_title(f'Bitmap {zoom}x', fontsize=12, fontweight='bold', color='red')
    ax.set_xticks([]); ax.set_yticks([])

axes[0][0].set_ylabel('VECTOR\n(Math)', fontsize=13, fontweight='bold', color='blue')
axes[1][0].set_ylabel('BITMAP\n(Pixels)', fontsize=13, fontweight='bold', color='red')
plt.suptitle('Scaling: Vector stays sharp, Bitmap gets blocky', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.show()

print('At 8x zoom:')
print('  Vector: Still a perfect circle (re-rendered from math)')
print('  Bitmap: Blocky mess (same pixels, just bigger)')

---
## 4. 详细对比

| 特征 | 位图 (Bitmap) | 矢量图 (Vector) |
|:---|:---|:---|
| **组成** | 像素网格 | 数学描述的几何形状 |
| **缩放** | 放大会失真 | 放大不失真 |
| **文件大小** | 通常较大 | 通常较小 |
| **真实感** | 很高（适合照片） | 较低（适合图形） |
| **编辑** | 逐像素修改 | 修改形状属性 |
| **颜色细节** | 可以每像素不同颜色 | 每个形状一种填充 |
| **格式** | .jpeg .png .bmp .gif | .svg .cgm .odg .ai |
| **适合** | 照片、复杂图像 | Logo、图标、图表、CAD |
| **打印** | 取决于 DPI | 任何尺寸都清晰 |

### 什么时候用哪种？

| 场景 | 选择 | 原因 |
|:---|:---|:---|
| 公司 Logo | 矢量 | 需要缩放到名片/广告牌等不同尺寸 |
| 数码照片 | 位图 | 照片天然就是像素数据 |
| 网页图标 | 矢量 (SVG) | 适配不同屏幕尺寸，文件小 |
| CAD 工程图 | 矢量 | 需要精确尺寸和缩放 |
| 游戏截图 | 位图 | 像素渲染的结果 |
| 漫画/插画 | 矢量 | 清晰的线条和填充 |

**注意：** 打印矢量图形时，通常需要先转换为位图（光栅化 rasterization），因为打印机是逐点打印的。

---
## 5. 练习题

1. 用你自己的话解释"绘图列表"是什么。
2. 一个矢量图文件包含 3 个圆和 2 个矩形。如果把它转换为 1000x1000 像素的位图（24位），位图文件大约多大？矢量文件呢？
3. 为什么照片不能用矢量格式保存？
4. 一个设计师要做一个 Logo，这个 Logo 要用在：名片(5cm)、网站(200px)、广告牌(5m)。为什么一定要用矢量图？
5. SVG 文件本质上是什么格式？这有什么好处？

In [ ]:
print("""Answers:

Q1: A drawing list is a set of instructions stored in a vector file.
    It describes each shape (circle, rectangle, line...) and its
    attributes (position, size, colour, border style...).
    The computer reads these instructions and draws the shapes.

Q2: Bitmap: 1000 x 1000 x 24 / 8 = 3,000,000 bytes = 3 MB
    Vector: Only stores ~5 shape definitions, probably < 1 KB!
    That is a 3000x difference!

Q3: A photo contains millions of pixels, each with a unique colour.
    There is no simple geometric shape that can describe a photo.
    You would need millions of tiny shapes, which defeats the purpose.
    Photos are inherently pixel-based data from camera sensors.

Q4: The Logo needs to work at 5cm, 200px, AND 5 metres.
    A bitmap designed for 5cm would be blurry at 5m.
    A bitmap designed for 5m would be an enormous file.
    A vector file works at ALL sizes - it is re-rendered each time.
    One file, infinite sizes, always sharp.

Q5: SVG is essentially an XML text file (like HTML).
    Benefits:
    - Can be edited with a text editor
    - Can be styled with CSS
    - Very small file size
    - Can be compressed further (gzip)
    - Can be indexed by search engines
    - Can be animated with JavaScript""")

---
**Next:** [04 - 声音数字化](04_声音数字化.ipynb)